# 🧠 05 — Neural Network on MNIST with PyTorch

Builds a **3-layer fully-connected network** from scratch using PyTorch.
Trains on MNIST (28×28 handwritten digits), plots loss & accuracy curves, and shows a confusion matrix.

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from visualizer import plot_training_curves, plot_confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# --- Data ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST('./data/raw', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data/raw', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds):,}  |  Test: {len(test_ds):,}')

In [ ]:
# --- Model ---
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

model     = MNISTNet().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
print(model)

In [ ]:
# --- Training loop ---
EPOCHS = 15
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * len(y_batch)
        t_correct += (out.argmax(1) == y_batch).sum().item()
        t_total   += len(y_batch)

    # Validate
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            out  = model(X_batch)
            loss = criterion(out, y_batch)
            v_loss    += loss.item() * len(y_batch)
            v_correct += (out.argmax(1) == y_batch).sum().item()
            v_total   += len(y_batch)

    train_losses.append(t_loss / t_total)
    val_losses.append(v_loss / v_total)
    train_accs.append(t_correct / t_total)
    val_accs.append(v_correct / v_total)
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS} | TrainLoss {train_losses[-1]:.4f} | ValLoss {val_losses[-1]:.4f} | ValAcc {val_accs[-1]:.4f}')

In [ ]:
# --- Curves & Confusion Matrix ---
plot_training_curves(train_losses, val_losses, train_accs, val_accs)

# Collect all predictions
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

cm = confusion_matrix(all_labels, all_preds)
plot_confusion_matrix(cm, class_names=[str(i) for i in range(10)],
                      title='MNIST Confusion Matrix')
print(f'Final Test Accuracy: {val_accs[-1]*100:.2f}%')

In [ ]:
# --- Save model ---
import os, torch
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/mnist_net.pt')
print('Model saved to models/mnist_net.pt')